# Unit 1 — Single AI Agent with Memory & Tools
## AI-Based Defect Reporting System
### Agentic AI & Automation — Symbiosis International University

**Learning Objectives (CO1):**
- Build and run a first simple AI agent
- Add SQLite-based session storage for persistent memory
- Integrate OpenAI/Gemini tools (web search)
- Monitor agent activity using traces


In [ ]:
import os, sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

from google import genai
client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
MODEL_NAME = 'gemini-2.5-flash'

print('✅ Gemini Client configured')
print(f'   Model: {MODEL_NAME}')

## Step 1: Build a Simple Agent (No Tools, No Memory)

In [ ]:
# Simplest possible AI agent
response = client.models.generate_content(
    model=MODEL_NAME,
    contents='Analyze this software defect: "The login form crashes with error 500 on Firefox". Give a brief technical analysis.'
)
print('🤖 Agent Response:')
print(response.text)

## Step 2: Add SQLite-Based Persistent Memory

In [ ]:
from app.memory.session_store import save_message, get_history, clear_session
import json

session_id = 'notebook-demo-01'

# Simulate a multi-turn conversation
save_message(session_id, 'user', 'Login page crashes on submit')
save_message(session_id, 'assistant', 'I will analyze this defect. Can you tell me which browser?')
save_message(session_id, 'user', 'It happens on Firefox 120')

# Retrieve history
history = get_history(session_id)
print('📚 Session Memory (SQLite):')
for msg in history:
    print(f'  [{msg["role"]}] {msg["content"]}')

print(f'\n✅ {len(history)} messages stored in SQLite')

## Step 3: Agent with Memory — Multi-Turn Query

In [ ]:
# Build contents list from SQLite history
from google.genai import types

contents = []
for msg in history:
    role = 'user' if msg['role'] == 'user' else 'model'
    contents.append(types.Content(role=role, parts=[types.Part(text=msg['content'])]))

# Add new question — agent remembers context!
contents.append(types.Content(role='user', parts=[types.Part(text='What do you think the root cause is?')]))

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=contents,
    config=types.GenerateContentConfig(
        system_instruction='You are a software defect analysis expert. Use conversation history.'
    )
)
print('🧠 Memory-Enabled Agent Response:')
print(response.text)

# Save the response to memory
save_message(session_id, 'assistant', response.text)
print('\n✅ Response saved to SQLite memory')

## Step 4: Add Web Search Tool (Tavily / Mock)

In [ ]:
from app.tools.search_tool import search_web

# Search for information about the defect
query = 'Firefox form submission 500 error fix'
print(f'🔍 Searching: "{query}"\n')
results = search_web(query)
print(results)

## Step 5: Full Agent with Memory + Tools (Tracing Enabled)

In [ ]:
# Use our complete analysis agent
from app.agents.analysis_agent import analyze_defect

print('='*60)
print('Running full Analysis Agent (Unit 1 complete demo)')
print('='*60)

result = analyze_defect(
    'The payment checkout button does nothing when clicked on Safari iOS 17',
    session_id='notebook-demo-full'
)

print(f'\n📊 Results:')
print(f'  Session ID: {result["session_id"]}')
print(f'  Memory entries: {result["history_length"]}')
print(f'  Tools called: {[t["tool"] for t in result["tool_calls_made"]]}')
print(f'\n🤖 Agent Response:\n{result["response"]}')

In [ ]:
# Cleanup
clear_session('notebook-demo-01')
print('✅ Unit 1 Demo Complete!')
print('   - Built single agent ✓')
print('   - Added SQLite memory ✓')
print('   - Integrated tools ✓')
print('   - Multi-turn queries ✓')
print('   - Agent tracing ✓')